In [ ]:
from pathlib import Path
import importlib
import sys

# Resolve project root robustly regardless of notebook launch location
PROJECT_ROOT = (
    Path.cwd() if (Path.cwd() / "config" / "config.yaml").exists() else Path.cwd().parent
).resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

load_module = importlib.import_module("data.load")
preprocess_module = importlib.import_module("data.preprocess")
health_index_module = importlib.import_module("features.health_index")

load_module = importlib.reload(load_module)
preprocess_module = importlib.reload(preprocess_module)
health_index_module = importlib.reload(health_index_module)

load_dataset = load_module.load_dataset
load_config = load_module.load_config
preprocess_train = preprocess_module.preprocess_train
preprocess_test = preprocess_module.preprocess_test
build_health_index = health_index_module.build_health_index

# Load
config = load_config(PROJECT_ROOT / "config" / "config.yaml")
config["dataset"]["raw_path"] = str(PROJECT_ROOT / "data" / "raw")
config["dataset"]["processed_path"] = str(PROJECT_ROOT / "data" / "processed")

train_raw, test_raw, _ = load_dataset(config)
train_proc, scaler, sensor_cols = preprocess_train(train_raw, config)
test_proc = preprocess_test(test_raw, config, scaler)

# Build HI
train_hi, test_hi, artifacts = build_health_index(train_proc, test_proc, config)

# --- Checks ---
assert train_hi.shape[0] == 20631
assert test_hi.shape[0] == 13096
assert "health_index" in train_hi.columns
assert "health_index" in test_hi.columns
assert train_hi["health_index"].between(0.0, 1.0).all()
assert test_hi["health_index"].between(0.0, 1.0).all()

mean_start = train_hi[train_hi["cycle"] == 1]["health_index"].mean()
last_cycles = train_hi.groupby("unit")["cycle"].max().reset_index(name="last_cycle")
last_points = train_hi.merge(last_cycles, on="unit")
mean_end = last_points[last_points["cycle"] == last_points["last_cycle"]]["health_index"].mean()
assert mean_start > mean_end, f"HI not trending down: start={mean_start:.3f}, end={mean_end:.3f}"
assert artifacts.explained_variance_ratio > 0.20
assert len(artifacts.sensor_loadings) == len(config["selected_sensors"])

print("✅ All checks passed")
print(f"   PC1 explained variance : {artifacts.explained_variance_ratio:.1%}")
print(f"   Inversion applied      : {artifacts.invert}")
print(f"   Mean HI at cycle 1     : {mean_start:.4f}")
print(f"   Mean HI at last cycle  : {mean_end:.4f}")
print("\nTop 5 sensor loadings (PC1):")
print(artifacts.sensor_loadings.head(5))